# 1. Fenêtrage temporel


In [75]:
import numpy as np
import scipy.signal as signal
import plotly.express as px
import pandas as pd # pour les dataframes
import sounddevice as sd # pour jouer les sons associés aux signaux

### Question 1

Fonctionnement : L'algorithme FFT standard renvoie les fréquences dans l'ordre à partir de 0. La fonction np.fft.fftshift() s'arrange placer la fréquence nulle au centre du tableau.
Intérêt : Facilite la lecture du spectre, en affichant l'intervalle -nue/2, nue/2 donc symétrie autour de 0
Transformation de l'axe : L'axe fréquentiel associé doit être défini entre -nue/2 et nue/2

### Question 2


In [76]:
K = 32

wB = signal.windows.boxcar(K)
wH = signal.windows.hann(K)

px.line(x= np.arange(K), y= wH, labels={"x":"Temps", "y":"Amplitude"}, title="Fenetre de Hann").show()
px.line(x= np.arange(K), y= wB, labels={"x":"Temps", "y":"Amplitude"}, title="Fenetre rectangulaire").show()

### Question 3

In [77]:
moy_B = np.mean(wB)
moy_H = np.mean(wH)

G = moy_B / moy_H

print("Moyenne Rectangulaire : ", moy_B)
print("Moyenne Hann : ", moy_H)
print("Facteur de Gain G : ", G)

Moyenne Rectangulaire :  1.0
Moyenne Hann :  0.484375
Facteur de Gain G :  2.064516129032258


### Question 4

In [78]:
N_fft = 4 * K

S_B = np.fft.fftshift(np.fft.fft(wB, n=N_fft))
S_H = np.fft.fftshift(np.fft.fft(wH, n=N_fft))

f_norm = np.linspace(-0.5, 0.5, N_fft)


df_freq = pd.DataFrame({
    "Fréquence Réduite": f_norm,
    "Rectangulaire (dB)": np.abs(S_B),
    "Hann (dB)": np.abs(S_H)
})

px.line(df_freq, x="Fréquence Réduite", y=["Rectangulaire (dB)", "Hann (dB)"], title="Comparaison des Spectres (Résolution vs Dynamique)", labels={"value": "Amplitude", "variable": "Signal"}).show()


### Question 5

In [79]:
def cos_discret(A, nu0, nue, phi, K):
    '''Fonction permettant de générer un cosinus discret. Elle prend en argument un facteur d'amplitude A,
    une fréquence propre nu0, une fréquence d'échatillonage nue, une phase phi, un nombre de points K.
    Cette fonction renvoie un vecteur k contenant les indices des échantillons de la séquence généré,
    un veccteur t contenant les instants auxquels ont été acquis les échantillons de la séquence générée,
    la séquence générée s'''
    k = np.arange(K)
    t = k / nue
    s = A * np.cos(2 * np.pi * nu0 * k * (1 / nue) + phi)
    return k, t, s

k1, t1, s1 = cos_discret(2, 82.35, 1000, 0, 256)
k2, t2, s2 = cos_discret(0.1, 100.15, 1000, 0, 256)

sB = s1 + s2

sH = sB * G * signal.windows.hann(256)

### Question 6

In [80]:
df_temps = pd.DataFrame({
    "Temps (s)": t1, 
    "Signal Brut (Rectangulaire)": sB,
    "Signal Fenêtré (Hann)": sH
})

px.line(df_temps, x="Temps (s)", y=["Signal Brut (Rectangulaire)", "Signal Fenêtré (Hann)"], title="Allure Temporelle des signaux sB et sH", labels={"value": "Amplitude", "variable": "Signal"}).show()

N_fft = 4 * 256
nue = 1000

S_B = np.fft.fftshift(np.fft.fft(sB, n=N_fft))
S_H = np.fft.fftshift(np.fft.fft(sH, n=N_fft))

f_axe = np.linspace(-nue/2, nue/2, N_fft)

df_freq = pd.DataFrame({
    "Fréquence (Hz)": f_axe,
    "Spectre Brut (Rectangulaire)": np.abs(S_B),
    "Spectre Fenêtré (Hann)": np.abs(S_H)
})

px.line(df_freq, x="Fréquence (Hz)", y=["Spectre Brut (Rectangulaire)", "Spectre Fenêtré (Hann)"], title="Comparaison des Spectres", labels={"value": "Amplitude", "variable": "Fenêtre"}).show()


### Question 7

In [ ]:
K = 64

k7, t7, s1 = cos_discret(2, 88.65, 1000, 0, K)
k70, t70, s2 = cos_discret(2, 100.15, 1000, 0, K)

sB = s1 + s2

w_rect = signal.windows.boxcar(K)
w_hann = signal.windows.hann(K)

sH = sB * G * w_hann

df_temps = pd.DataFrame({
    "Temps (s)": t7, 
    "Signal Brut (Rectangulaire)": sB,
    "Signal Fenêtré (Hann)": sH
})

px.line(df_temps, x="Temps (s)", y=["Signal Brut (Rectangulaire)", "Signal Fenêtré (Hann)"], title="Allure Temporelle (K=64)", labels={"value": "Amplitude", "variable": "Signal"}).show()

N_fft = 4 * K 
S_B = np.fft.fftshift(np.fft.fft(sB, n=N_fft))
S_H = np.fft.fftshift(np.fft.fft(sH, n=N_fft))

f_axe = np.linspace(-nue/2, nue/2, N_fft)

df_freq = pd.DataFrame({
    "Fréquence (Hz)": f_axe,
    "Spectre Brut (Rectangulaire)": np.abs(S_B),
    "Spectre Fenêtré (Hann)": np.abs(S_H)
})

px.line(df_freq, x="Fréquence (Hz)", y=["Spectre Brut (Rectangulaire)", "Spectre Fenêtré (Hann)"], title="Comparaison des Spectres (Problème de Résolution)", labels={"value": "Amplitude", "variable": "Fenêtre"}).show()


# 2. Analyse temps-fréquence

## 2.1. Création du signal


### Question 1

In [85]:
nue = 1000
Duree = 1 
K_seg = int(Duree * nue) 

t_seg = np.arange(K_seg) / nue

nu1 = 100
nu2 = 50
nu3 = 200
nu4 = 200

s1 = np.cos(2 * np.pi * nu1 * t_seg)
s2 = np.cos(2 * np.pi * nu2 * t_seg) + np.cos(2 * np.pi * nu3 * t_seg)
s3 = np.cos(2 * np.pi * nu4 * t_seg**2)


### Question 2

In [86]:
s = np.concatenate((s1, s2, s3))
t_total = np.arange(len(s)) / nue

px.line(x=t_total, y=s, title="Séquence finale en fonction du temps", labels={"x": "Temps (s)", "y": "Amplitude"}).show()

## 2.2. Spectrogramme


### Question 1

In [87]:
N_fft_total = len(s)
S_total = np.fft.fftshift(np.fft.fft(s))
f_total = np.linspace(-nue/2, nue/2, N_fft_total)

px.line(x=f_total, y=np.abs(S_total), title="Spectre Global de s (TFD classique)",labels={"x": "Fréquence (Hz)", "y": "Module"}).show()

### Question 5

In [94]:
K = 64

wB = signal.windows.boxcar(K)
wH = signal.windows.hann(K)

SFT_B = signal.ShortTimeFFT(win=wB, hop=K, fs=nue, mfft=K)
SFT_H = signal.ShortTimeFFT(win=wH, hop=K, fs=nue, mfft=K)

nu_spec_B = SFT_B.f
nu_spec_H = SFT_H.f
t_spec_B = SFT_B.t(len(s))
t_spec_H = SFT_H.t(len(s))

spectrogram_B = SFT_B.stft(s)
spectrogram_H = SFT_H.stft(s)

# Affichage d'un spectrogramme
px.imshow(np.abs(spectrogram_B), x = t_spec_B, y = nu_spec_B, origin='lower', aspect='auto', labels={'x': 'Temps (s)', 'y':'Fréquence (Hz)', 'color': 'Magnitude'}, title='Spectrogramme rectagulaire').show()
px.imshow(np.abs(spectrogram_H), x = t_spec_H, y = nu_spec_H, origin='lower', aspect='auto', labels={'x': 'Temps (s)', 'y':'Fréquence (Hz)', 'color': 'Magnitude'}, title='Spectrogramme Hann').show()